# DeepEmbryo — 5. Gün Embriyo Kalite Değerlendirme

**Model:** ConvNeXt-Base (ImageNet-22k pretrained) — transfer learning + 5-fold stratified cross-validation.

**Donanım:** Colab Free Tier T4 GPU (Runtime → Change runtime type → T4 GPU).

**Beklenen toplam süre:** 1–2 saat (kurulum + 5-fold eğitim + değerlendirme + XAI).

---

## Başlamadan önce
Lokalde `code-base` klasörünü `code-base.zip` olarak sıkıştırın (içinde `src/`, `app/`, `notebooks/`, `requirements.txt`, `EMBRIO GRADE DATASET/` vb. olmalı). Sonra Colab'de **soldaki dosya simgesine** tıklayıp zip'i `/content/` altına sürükleyin (`/content/code-base.zip`).

## 1. Zip'i çıkar ve proje klasörüne geç

In [ ]:
import os
import zipfile
import shutil

zip_path = '/content/code-base.zip'
extract_path = '/content/code-base'

print('Zip var mı?', os.path.exists(zip_path))
if os.path.exists(zip_path):
    print('Zip boyutu:', os.path.getsize(zip_path), 'byte')
    print('Geçerli zip mi?', zipfile.is_zipfile(zip_path))

if os.path.exists(zip_path) and zipfile.is_zipfile(zip_path):
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print('Zip çıkarıldı:', extract_path)
else:
    raise SystemExit('Zip bulunamadı veya geçerli değil. Soldaki dosya panelinden code-base.zip dosyasını /content/ altına yükleyin.')

print('\n/content içeriği:')
!ls -la /content
print('\nÇıkan klasörler:')
!find /content/code-base -maxdepth 3 -type d

In [ ]:
# Bazı zip'ler tek bir kök klasör içerir (ör. /content/code-base/code-base/...).
# Eğer durum buysa, gerçek proje köküne otomatik geçelim.
from pathlib import Path

candidates = [Path(extract_path)]
for child in Path(extract_path).iterdir():
    if child.is_dir():
        candidates.append(child)

PROJECT_ROOT = next(c for c in candidates if (c / 'src' / 'config.py').exists())
print('Proje kökü:', PROJECT_ROOT)
%cd {PROJECT_ROOT}
!ls

## 2. Bağımlılıkları kur

In [ ]:
!pip install -q -r requirements.txt

## 3. GPU & ortam kontrolü

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Veri seti sağlık kontrolü

In [ ]:
from src.data import EmbryoDataset, build_transforms
from src import config as cfg
import numpy as np

ds = EmbryoDataset(transform=build_transforms(train=False))
labels = ds.get_labels()
print('Toplam görüntü:', len(ds))
for i, c in enumerate(cfg.CLASSES):
    print(f'  {c}: {(labels == i).sum()}')

img, lbl, path = ds[0]
print('Örnek tensor şekli:', img.shape, '| etiket:', cfg.IDX_TO_CLASS[lbl])

## 5. Model forward sanity check

In [ ]:
from src.model import build_model, count_trainable

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model().to(device)
print('Toplam parametre:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')
print('Eğitilebilir parametre:', count_trainable(model) / 1e6, 'M')

with torch.no_grad():
    out = model(torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=device))
print('Çıktı şekli:', out.shape, '(beklenen: torch.Size([2, 4]))')

## 6. Tam 5-fold eğitim

**Süre:** Yaklaşık 1–1.5 saat T4 GPU'da.

In [ ]:
from src.train import run_kfold

fold_results = run_kfold(device=device)
print('\n=== Fold sonuçları ===')
for fr in fold_results:
    print(f"fold {fr['fold']}: best_val_loss={fr['best_val_loss']:.4f}")

## 7. Değerlendirme: confusion matrix, classification report, ROC, history grafikleri

In [ ]:
from src.evaluate import evaluate_all_folds, select_best_fold, export_final_model
import json

summary = evaluate_all_folds(fold_results, device=device)
print(json.dumps(summary['fold_mean'], indent=2))
print('Std:')
print(json.dumps(summary['fold_std'], indent=2))

best_fold = select_best_fold(summary['per_fold'])
print(f'\nEn iyi fold: {best_fold}')
final_path = export_final_model(best_fold)
print(f'Final model: {final_path}')

## 8. Learning curve (doc §4.1)

In [ ]:
from src.evaluate import learning_curve

lc_df = learning_curve(device=device, fractions=(0.25, 0.5, 0.75, 1.0), epochs=15)
lc_df

## 9. Grad-CAM örnek galerisi (doc §5.1)

In [ ]:
from src.infer import EmbryoPredictor, export_gradcam_samples

predictor = EmbryoPredictor(checkpoint_path=final_path, device=device)
export_gradcam_samples(predictor, n_per_class=3)
print('Grad-CAM örnekleri ->', cfg.GRADCAM_DIR)
for f in sorted(os.listdir(cfg.GRADCAM_DIR))[:12]:
    print(' ', f)

## 10. Morfolojik özellik raporu (doc §5.3)

In [ ]:
from src.morphology import build_morphology_report

morph_df = build_morphology_report(
    predictor.model,
    predictions_csv=cfg.OUTPUT_DIR / 'predictions.csv',
    device=device,
)
morph_df.head()

## 11. Tek görüntü tahmin (uyarı sistemi demosu — doc §5.2)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

sample_path = next((cfg.DATA_DIR / '4AA').glob('*.bmp'))
out_cam = cfg.GRADCAM_DIR / 'demo_single.png'
pred = predictor.predict_single(sample_path, gradcam_save_path=out_cam)

print(f'Tahmin: {pred.predicted_class}  (güven={pred.confidence:.2f})')
if pred.warning:
    print('UYARI:', pred.warning)
else:
    print('Yüksek güven — uyarı yok.')
print('Olasılıklar:', pred.probabilities)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(sample_path)); axes[0].set_title('Orijinal'); axes[0].axis('off')
axes[1].imshow(Image.open(out_cam)); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 13. Çıktıların listesi

Tüm artefaktlar `outputs/` altında (Colab oturumu kapanınca silinir — bir sonraki adımda zip'leyip indireceğiz).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_preds = pd.read_csv(cfg.OUTPUT_DIR / 'predictions.csv')
df_preds['is_correct'] = (df_preds['y_true'] == df_preds['y_pred']).astype(int)
df_preds['warning'] = (df_preds['confidence'] < cfg.CONFIDENCE_THRESHOLD).astype(int)

n = len(df_preds)
mean_conf = df_preds['confidence'].mean()
median_conf = df_preds['confidence'].median()
std_conf = df_preds['confidence'].std()
warning_rate = df_preds['warning'].mean() * 100
overall_acc = df_preds['is_correct'].mean() * 100

print(f'Toplam tahmin: {n}')
print(f'Genel doğruluk: {overall_acc:.1f}%')
print(f'Ortalama güven: {mean_conf:.3f}  (medyan {median_conf:.3f}, std {std_conf:.3f})')
print(f'Uyarı tetiklenen tahminler (güven < {cfg.CONFIDENCE_THRESHOLD}): {warning_rate:.1f}%')

print('\n--- Doğru vs yanlış tahminlerin ortalama güveni ---')
by_correct = df_preds.groupby('is_correct')['confidence'].agg(['mean', 'std', 'count'])
by_correct.index = ['Yanlış (0)', 'Doğru (1)']
print(by_correct.round(3))

print('\n--- Tahmin edilen sınıfa göre ortalama güven ---')
by_pred = df_preds.groupby('y_pred')['confidence'].agg(['mean', 'std', 'count']).round(3)
print(by_pred)

print('\n--- Gerçek sınıfa göre ortalama güven ---')
by_true = df_preds.groupby('y_true')['confidence'].agg(['mean', 'std', 'count']).round(3)
print(by_true)

## 14. Teslim paketini hazırla ve indir

## 12. Çıktıların listesi

Tüm artefaktlar `outputs/` altında (Colab oturumu kapanınca silinir — bir sonraki adımda zip'leyip indireceğiz).

In [ ]:
import subprocess
print(subprocess.check_output(['ls', '-R', str(cfg.OUTPUT_DIR)]).decode()[:4000])

## 13. Teslim paketini hazırla ve indir

In [ ]:
!python package.py

In [ ]:
# DeepEmbryo_Teslim.zip'i bilgisayara indir.
from google.colab import files
files.download(str(PROJECT_ROOT / 'DeepEmbryo_Teslim.zip'))

In [ ]:
# (Opsiyonel) Sadece outputs/ klasörünü ham haliyle de zip'leyip indir.
shutil.make_archive('/content/outputs', 'zip', root_dir=str(cfg.OUTPUT_DIR))
files.download('/content/outputs.zip')